Se debe realizar una evaluación de Clusters

    1. Calcular métricas de calidad: Silhouette y Davies-Bouldin.

    2. Comparar resultados entre K-means (numérico) y Agglomerative (Gower).

    3. Documentar similitudes y diferencias

## Paso 1: Preparación de datos y cálculo de K-means

Aplicamos K-means sobre variables numéricas (similar al Ejercicio 4).


In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Configuración
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# Cargar dataset
df = cargar_dataset_adult()

# Tomar muestra (5000 registros para consistencia con Ejercicio 5)
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42).copy()

print("="*60)
print("EJERCICIO 6: EVALUACIÓN Y COMPARACIÓN DE CLUSTERS")
print("="*60)
print(f"\nMuestra utilizada: {sample_size} observaciones")

# Variables para K-means (solo numéricas)
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
X = df_sample[numeric_cols].copy()

# Normalizar datos para K-means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n" + "-"*60)
print("PARTE 1: K-MEANS (SOLO VARIABLES NUMÉRICAS)")
print("-"*60)

# Aplicar K-means con distintos valores de k
k_range = range(2, 11)
kmeans_results = {
    'k': [],
    'silhouette': [],
    'davies_bouldin': []
}

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    silhouette = silhouette_score(X_scaled, labels)
    davies_bouldin = davies_bouldin_score(X_scaled, labels)
    
    kmeans_results['k'].append(k)
    kmeans_results['silhouette'].append(silhouette)
    kmeans_results['davies_bouldin'].append(davies_bouldin)

# Determinar k óptimo para K-means
kmeans_best_k_silhouette = k_range[np.argmax(kmeans_results['silhouette'])]
kmeans_best_k_db = k_range[np.argmin(kmeans_results['davies_bouldin'])]

print(f"\nK-means - Métricas de calidad:")
kmeans_df = pd.DataFrame(kmeans_results)
print(kmeans_df.to_string(index=False))

print(f"\nK-means - K óptimo según Silhouette: {kmeans_best_k_silhouette}")
print(f"K-means - K óptimo según Davies-Bouldin: {kmeans_best_k_db}")


## Paso 2: Cálculo de Agglomerative Clustering con Gower

Aplicamos Agglomerative Clustering con matriz de distancias de Gower (similar al Ejercicio 5).


In [ ]:
print("\n" + "-"*60)
print("PARTE 2: AGGLOMERATIVE CLUSTERING (DISTANCIA DE GOWER)")
print("-"*60)

# Identificar tipos de variables (sin fnlwgt)
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# Manejo de la Variable Ordinal 'education'
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample_gower = df_sample.copy()
df_sample_gower['education'] = (df_sample_gower['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}

# Calcular matriz de Gower
print("\nCalculando matriz de distancias de Gower...")
df_num = df_sample_gower[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample_gower[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample_gower[ordinal_cols].to_numpy(dtype=np.float64)

numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim
is_not_nan = ~df_sample_gower.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)

np.fill_diagonal(gower_matrix, 0)

# Aplicar Agglomerative Clustering con distintos valores de k
agglomerative_results = {
    'k': [],
    'silhouette': [],
    'davies_bouldin': []
}

for k in k_range:
    agg = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    labels = agg.fit_predict(gower_matrix)
    
    silhouette = silhouette_score(gower_matrix, labels, metric='precomputed')
    
    # Davies-Bouldin con variables numéricas originales
    df_temp = df.sample(n=sample_size, random_state=42)[numeric_cols]
    davies_bouldin = davies_bouldin_score(df_temp, labels)
    
    agglomerative_results['k'].append(k)
    agglomerative_results['silhouette'].append(silhouette)
    agglomerative_results['davies_bouldin'].append(davies_bouldin)

# Determinar k óptimo para Agglomerative
agg_best_k_silhouette = k_range[np.argmax(agglomerative_results['silhouette'])]
agg_best_k_db = k_range[np.argmin(agglomerative_results['davies_bouldin'])]

print(f"\nAgglomerative - Métricas de calidad:")
agg_df = pd.DataFrame(agglomerative_results)
print(agg_df.to_string(index=False))

print(f"\nAgglomerative - K óptimo según Silhouette: {agg_best_k_silhouette}")
print(f"Agglomerative - K óptimo según Davies-Bouldin: {agg_best_k_db}")


## Paso 3: Comparación directa de resultados

Comparamos las métricas de ambos métodos para cada valor de k.


In [ ]:
print("\n" + "="*60)
print("COMPARACIÓN DE MÉTRICAS: K-MEANS VS AGGLOMERATIVE")
print("="*60)

# Crear tabla comparativa
comparison_df = pd.DataFrame({
    'k': kmeans_results['k'],
    'K-means Silhouette': kmeans_results['silhouette'],
    'Agglomerative Silhouette': agglomerative_results['silhouette'],
    'K-means Davies-Bouldin': kmeans_results['davies_bouldin'],
    'Agglomerative Davies-Bouldin': agglomerative_results['davies_bouldin']
})

print("\nTabla Comparativa de Métricas:")
print(comparison_df.to_string(index=False))

# Calcular diferencias promedio
avg_silhouette_diff = np.mean(np.array(kmeans_results['silhouette']) - np.array(agglomerative_results['silhouette']))
avg_db_diff = np.mean(np.array(kmeans_results['davies_bouldin']) - np.array(agglomerative_results['davies_bouldin']))

print(f"\n\nDiferencias promedio:")
print(f"  Silhouette (K-means - Agglomerative): {avg_silhouette_diff:.4f}")
print(f"  Davies-Bouldin (K-means - Agglomerative): {avg_db_diff:.4f}")

# Determinar qué método es mejor en promedio
if avg_silhouette_diff > 0:
    print("\n  -> K-means tiene mejor Silhouette promedio (más alto es mejor)")
else:
    print("\n  -> Agglomerative tiene mejor Silhouette promedio (más alto es mejor)")

if avg_db_diff < 0:
    print("  -> K-means tiene mejor Davies-Bouldin promedio (más bajo es mejor)")
else:
    print("  -> Agglomerative tiene mejor Davies-Bouldin promedio (más bajo es mejor)")


## Paso 4: Documentación de similitudes y diferencias

Análisis cualitativo de las características de cada método.


In [ ]:
print("\n" + "="*60)
print("SIMILITUDES Y DIFERENCIAS ENTRE MÉTODOS")
print("="*60)

print("\n[1] SIMILITUDES:")
print("-" * 60)
print("  1. Ambos métodos son algoritmos de clustering no supervisado")
print("  2. Ambos requieren especificar el número de clusters k de antemano")
print("  3. Ambos buscan agrupar observaciones similares en clusters")
print("  4. Las métricas de evaluación (Silhouette y Davies-Bouldin) son aplicables a ambos")
print("  5. El k óptimo varía según la métrica utilizada en ambos casos")

print("\n[2] DIFERENCIAS CLAVE:")
print("-" * 60)

print("\nA. TIPO DE VARIABLES:")
print("  K-means:")
print("    - Solo trabaja con variables NUMÉRICAS")
print("    - Usa 5 variables: age, capital-gain, capital-loss, hours-per-week, education-num")
print("    - Ignora información categórica valiosa (educación, estado civil, ocupación, etc.)")
print("  ")
print("  Agglomerative (Gower):")
print("    - Trabaja con TODOS los tipos de variables (numéricas, categóricas, ordinales)")
print(f"    - Usa {len(numeric_cols)} numéricas + {len(categorical_cols)} categóricas + {len(ordinal_cols)} ordinal")
print("    - Aprovecha toda la información disponible del dataset")

print("\nB. MÉTRICA DE DISTANCIA:")
print("  K-means:")
print("    - Usa distancia EUCLIDIANA en espacio normalizado")
print("    - Asume que todas las variables son continuas y comparables")
print("    - Requiere normalización previa (StandardScaler)")
print("  ")
print("  Agglomerative (Gower):")
print("    - Usa distancia de GOWER (métrica mixta)")
print("    - Maneja apropiadamente diferentes tipos de variables")
print("    - No requiere normalización (Gower ya normaliza internamente)")

print("\nC. ALGORITMO DE AGRUPAMIENTO:")
print("  K-means:")
print("    - Algoritmo iterativo basado en centroides")
print("    - Busca minimizar la varianza intra-cluster")
print("    - Más rápido computacionalmente")
print("    - Puede quedar atrapado en mínimos locales")
print("  ")
print("  Agglomerative:")
print("    - Algoritmo jerárquico basado en matriz de distancias")
print("    - Construye árbol de jerarquías (dendrograma)")
print("    - Más lento (requiere calcular matriz n×n)")
print("    - Resultado determinístico (no hay aleatoriedad)")

print("\nD. REQUISITOS COMPUTACIONALES:")
print("  K-means:")
print(f"    - Complejidad: O(n·k·i) donde i = iteraciones")
print(f"    - Memoria: O(n·d) donde d = dimensiones")
print("    - Eficiente para datasets grandes")
print("  ")
print("  Agglomerative (Gower):")
print(f"    - Complejidad: O(n²) para calcular matriz + O(n²·log n) para clustering")
print(f"    - Memoria: O(n²) para matriz de distancias")
print("    - Limitado por el tamaño de la matriz (usamos muestra de {sample_size})")

print("\nE. K ÓPTIMO ENCONTRADO:")
print(f"  K-means:")
print(f"    - Según Silhouette: k = {kmeans_best_k_silhouette} (score: {max(kmeans_results['silhouette']):.4f})")
print(f"    - Según Davies-Bouldin: k = {kmeans_best_k_db} (score: {min(kmeans_results['davies_bouldin']):.4f})")
print(f"  ")
print(f"  Agglomerative:")
print(f"    - Según Silhouette: k = {agg_best_k_silhouette} (score: {max(agglomerative_results['silhouette']):.4f})")
print(f"    - Según Davies-Bouldin: k = {agg_best_k_db} (score: {min(agglomerative_results['davies_bouldin']):.4f})")


print("\n" + "="*60)
print("="*60)
